# Étape 1 — OCR Vision

Transcrit les PDF du dossier Drive en fichiers `<Livre>_OCR.txt`, page par page.

**Le modèle ne corrige rien.** Il transcrit ce qu'il voit. C'est l'étape 2 qui corrigera, et c'est parce que `OCR.txt` reste brut qu'il pourra servir de référence à l'étape 3.

---

**Ce notebook n'est qu'une interface.** Toute la logique vit dans le paquet
`theatre_editor`. On y monte le Drive, on installe les dépendances, on surcharge
éventuellement la configuration, puis on lance l'étape.

**Cette étape est reprenable.** Si Colab coupe, relancez la cellule
d'exécution : le travail déjà validé ne sera pas refait, et vous ne repaierez
aucun appel.

## 1. Dépendances et montage du Drive

In [ ]:
# Installation des dépendances du pipeline.
!pip install -q -U openai pymupdf python-docx

from google.colab import drive

drive.mount("/content/drive")

## 2. Récupération du code

Le dépôt [`elyeskaak/texte_troupe_theatre`](https://github.com/elyeskaak/texte_troupe_theatre)
est **public** : rien à configurer, la cellule suivante suffit. Elle récupère la
dernière version du code à chaque exécution.

> **Si vous repassiez le dépôt en privé**, il faudrait un jeton d'accès :
> GitHub → *Settings* → *Developer settings* → *Personal access tokens* →
> *Fine-grained tokens*, avec **Contents : Read-only** sur ce seul dépôt. Puis
> l'enregistrer dans les Secrets de Colab sous le nom `GITHUB_TOKEN`. La
> cellule le détecte et l'utilise automatiquement — aucune modification à faire.

In [ ]:
# --- Option A : récupération depuis GitHub ------------------------------
DEPOT_COMPTE = "elyeskaak"
DEPOT_NOM = "texte_troupe_theatre"
DOSSIER_PROJET = f"/content/{DEPOT_NOM}"

import os
import subprocess
import sys

# Le jeton est OPTIONNEL : inutile sur un dépôt public, utilisé
# automatiquement s'il est présent. Ainsi la cellule fonctionne dans les deux
# cas, sans qu'il faille se souvenir de la visibilité du dépôt.
jeton = None

try:
    from google.colab import userdata

    jeton = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

# Quand un jeton est utilisé, l'URL le contient : elle ne doit JAMAIS être
# affichée ni figurer dans un message d'erreur. Les sorties de git sont donc
# capturées, jamais relayées.
if jeton:
    url = f"https://{jeton}@github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"
else:
    url = f"https://github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"

if os.path.isdir(DOSSIER_PROJET):
    commande = ["git", "-C", DOSSIER_PROJET, "pull", "--quiet"]
else:
    commande = ["git", "clone", "--quiet", url, DOSSIER_PROJET]

resultat = subprocess.run(commande, capture_output=True, text=True)

if resultat.returncode != 0:
    raise RuntimeError(
        "Récupération du code impossible.\n"
        f"Vérifiez que le dépôt {DEPOT_COMPTE}/{DEPOT_NOM} est accessible.\n"
        "S'il est privé, ajoutez un secret GITHUB_TOKEN dans Colab."
    )

if DOSSIER_PROJET not in sys.path:
    sys.path.insert(0, DOSSIER_PROJET)

print("Code récupéré :", DOSSIER_PROJET)
print("Jeton GitHub  :", "utilisé" if jeton else "non nécessaire (dépôt public)")

In [ ]:
# --- Option B : le dossier theatre_editor/ est sur votre Drive ---------
# Décommentez ces lignes et ajustez le chemin, puis n'exécutez PAS l'option A.

# import sys
# DOSSIER_PROJET = "/content/drive/MyDrive/texte_troupe_theatre"
# if DOSSIER_PROJET not in sys.path:
#     sys.path.insert(0, DOSSIER_PROJET)

## 3. Migration des livres déjà traités

**À lancer une fois**, si vous avez utilisé le pipeline avant le changement de
disposition du Drive.

Le dossier principal ne contient désormais que vos PDF et les DOCX produits ;
tout le travail intermédiaire est rangé dans `temp/<Nom du livre>/`.

Cette cellule déplace les fichiers existants vers la nouvelle disposition. C'est
un simple déplacement : **rien n'est perdu et rien n'est repayé**. Sans elle, vos
transcriptions deviendraient invisibles et seraient refaites.

L'opération ne fait rien si elle a déjà été effectuée.

In [ ]:
from theatre_editor.utils import io

a_migrer = io.livres_a_migrer(config.DOSSIER_DRIVE)

if not a_migrer:
    print("Rien à migrer : la disposition est déjà à jour.")
else:
    for nom in a_migrer:
        print(nom)
        for ligne in io.migrer_livre(nom, config.DOSSIER_DRIVE):
            print("   ", ligne)

    for nom in io.migrer_journaux(config.DOSSIER_DRIVE):
        print("journal :", nom)

## 4. Configuration

`config.py` porte toutes les valeurs par défaut. Les surcharges ci-dessous ne
valent que pour cette session : elles ne modifient pas le fichier.

**Vérifiez le dossier de travail** avant de continuer.

In [ ]:
from pathlib import Path

from theatre_editor import config

# Dossier Drive contenant les PDF et recevant toutes les sorties.
config.DOSSIER_DRIVE = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")

print("Dossier de travail :", config.DOSSIER_DRIVE)
print("Existe             :", config.DOSSIER_DRIVE.is_dir())

## 5. Clé API

In [ ]:
# La clé API est lue depuis les Secrets de Colab.
#
#   panneau latéral « 🔑 Secrets » → ajouter OPENAI_API_KEY
#   → activer « Accès au notebook »
#
# Ainsi la clé n'apparaît jamais dans le notebook ni dans ses sorties.

from theatre_editor.utils import io

try:
    io.charger_cle_api()
    print("Clé API trouvée.")
except RuntimeError as erreur:
    print(erreur)

## 6. Vérification des modèles

In [ ]:
# Contrôle que les identifiants de config.py existent bien sur ce compte.
# Deux secondes ici évitent de découvrir une faute de frappe après trois
# heures de traitement.

from theatre_editor.utils import api

api.verifier_modeles_configures()

## 7. Aperçu du travail à faire

Liste les PDF trouvés et l'avancement de chacun, sans lancer aucun appel.

In [ ]:
from theatre_editor.utils import io

for chemin in io.lister_pdf(config.DOSSIER_DRIVE):
    nom = io.nom_livre_depuis_pdf(chemin)
    chemins = io.resoudre_chemins(nom, chemin.parent)

    faites = sum(
        1
        for fichier in sorted(chemins.dossier_pages.glob("page_*.json"))
        if io.unite_terminee(fichier)
    ) if chemins.dossier_pages.is_dir() else 0

    taille_mo = chemin.stat().st_size / (1024 * 1024)
    print(f"{nom:<40} {taille_mo:>6.1f} Mo   {faites} page(s) déjà transcrite(s)")

## 8. Combien de pages seront réellement facturées ?

**Cette cellule ne consomme aucun jeton.**

Beaucoup de PDF ont déjà été passés à l'OCR par un scanner ou par Acrobat, et
portent donc une couche texte. Quand elle est de bonne qualité, le pipeline la
réutilise telle quelle : aucun appel API pour ces pages.

Mais une couche texte n'est pas forcément bonne — accents dépouillés, ligatures,
ordre de lecture faux. S'en servir à tort dégraderait tout le livre, puisque
l'étape 2 a pour consigne de ne pas réécrire l'auteur. Les contrôles sont donc
sévères, et le doute renvoie à l'OCR Vision.

Le diagnostic indique, pour chaque livre, combien de pages sont gratuites et
**pourquoi** les autres ne le sont pas.

In [ ]:
from theatre_editor import ocr

diagnostics = ocr.diagnostiquer_couches_texte(config.DOSSIER_DRIVE)

### Ajuster la stratégie

`config.STRATEGIE_COUCHE_TEXTE` accepte trois valeurs.

| Valeur | Effet |
|---|---|
| `"auto"` | couche texte utilisée si elle passe les contrôles — **recommandé** |
| `"jamais"` | toujours l'OCR Vision, même sur un PDF déjà OCRisé |
| `"toujours"` | couche texte utilisée dès qu'elle existe, sans contrôle |

`"toujours"` est à réserver aux PDF dont vous connaissez la provenance et la
qualité. Sur un fichier douteux, il produirait un livre dégradé sans le signaler.

Si le diagnostic écarte beaucoup de pages pour un motif qui vous paraît trop
strict, vous pouvez relâcher le seuil correspondant — mais relisez alors un
extrait de la couche texte avant de lancer le livre entier.

In [ ]:
config.STRATEGIE_COUCHE_TEXTE = "auto"

# Seuils de qualité, à ne relâcher qu'en connaissance de cause :
# config.MIN_CARACTERES_COUCHE_TEXTE = 200
# config.MIN_RATIO_ACCENTS = 0.005

print("Stratégie :", config.STRATEGIE_COUCHE_TEXTE)

### Inspecter une couche texte avant de lui faire confiance

Affiche ce que le PDF contient déjà pour une page donnée. À faire au moins une
fois sur un livre dont vous ne connaissez pas l'origine.

In [ ]:
NOM_LIVRE = diagnostics[0].nom if diagnostics else None
NUMERO_PAGE = 5

if NOM_LIVRE:
    chemins = io.resoudre_chemins(NOM_LIVRE, config.DOSSIER_DRIVE)
    document = ocr.ouvrir_pdf(chemins.pdf)

    try:
        numero = min(NUMERO_PAGE, document.page_count)
        texte, raisons = ocr.evaluer_page_couche_texte(
            document.load_page(numero - 1)
        )
    finally:
        document.close()

    print(f"{NOM_LIVRE} — page {numero}")
    print("=" * 72)
    print(f"caractères extraits : {len(texte)}")
    print(f"retenue             : {ocr.couche_texte_retenue(texte, raisons)}")

    if raisons:
        print("motifs de refus     :")
        for raison in raisons:
            print("   -", raison)

    print("=" * 72)
    print(texte[:1200] if texte else "(aucune couche texte)")

## 9. Essai sur les premières pages

**À faire sur tout nouveau livre.** Éprouver les quatre étapes sur dix pages
coûte quelques centimes et révèle les mauvaises surprises — modèle qui refuse la
vision, couche texte trompeuse, structure mal reconnue — avant d'engager
trois cents pages.

Les pages transcrites pendant l'essai sont **conservées et réutilisées** lors du
passage complet : rien n'est perdu, rien n'est repayé.

Enchaînez ensuite les notebooks 02, 03 et 04 : ils travaillent depuis `OCR.txt`,
qui ne contiendra que les pages retenues. Aucun réglage à y reporter.

In [ ]:
# Nombre de pages à traiter par PDF. None = livre entier.
config.LIMITE_PAGES = 10

print("Limite de pages :", config.LIMITE_PAGES)

### Passer au livre entier

Remettez `LIMITE_PAGES` à `None` et relancez la cellule de lancement. Seules les
pages manquantes seront transcrites.

Un point de vigilance, désormais géré par le code. Avec des blocs de 8 pages, un
essai de 10 pages produit un « bloc 2 » couvrant les pages 9 et 10, tandis que le
livre entier attend un bloc 2 couvrant les pages 9 à 16. L'étape 2 compare donc
les frontières enregistrées à celles recalculées, et **réédite tout bloc dont les
frontières ont changé** — sans quoi les pages 11 à 16 disparaîtraient
silencieusement.

Vous verrez alors s'afficher, ce qui est normal :

```
[ALERTE]  bloc 2 : frontières changées (pages 9–10 → 9–16), réédition
```

In [ ]:
# Décommentez pour traiter le livre entier :

# config.LIMITE_PAGES = None

## 10. Lancement

Reprenable : relancez cette cellule autant de fois qu'il le faut.

Comptez environ 3 à 6 secondes par page. Un livre de 300 pages demande donc
entre 20 et 30 minutes, et la session Colab peut couper d'ici là — ce n'est pas
un problème.

In [ ]:
from theatre_editor import ocr

resultats = ocr.executer(config.DOSSIER_DRIVE)

## 11. Contrôle du résultat

Affiche le début de chaque fichier produit, et signale les pages en échec.

In [ ]:
for resultat in resultats:
    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)
    print("=" * 72)
    print(resultat.nom, "—", resultat.statut)
    print("=" * 72)

    if resultat.numeros_echoues:
        print("Pages en échec :", resultat.numeros_echoues)
        print("Relancez la cellule 7 pour les reprendre.")
        print()

    if chemins.ocr.exists():
        print(io.lire_texte(chemins.ocr)[:1200])

## 12. Journal

In [ ]:
# Journal détaillé de l'étape : un enregistrement par appel API, avec sa
# date, son modèle, son response_id, sa durée et sa consommation de jetons.

import json

chemin = config.DOSSIER_DRIVE / config.NOM_JOURNAL.format(etape="ocr")

if chemin.exists():
    journal = json.loads(chemin.read_text(encoding="utf-8"))
    print("Dernière exécution :", journal["derniere_execution"])
    print("Configuration      :", json.dumps(journal["configuration"], ensure_ascii=False))
    print()

    for nom, bilan in journal["livres"].items():
        print(f"{nom} : {json.dumps(bilan, ensure_ascii=False)}")

    jetons = sum(
        (appel.get("tokens_entree") or 0) + (appel.get("tokens_sortie") or 0)
        for appel in journal["appels"]
    )
    print()
    print(f"{len(journal['appels'])} appel(s) journalisé(s), {jetons:,} jetons".replace(",", " "))
else:
    print("Aucun journal : l'étape n'a pas encore été lancée.")